# Imports

In [1]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v


PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [2]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine            import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools          import Nevergrad_Spice_Bode_Optimizer
from symxplorer.designer_tools.utils    import Frequency_Weight
from symxplorer.designer_tools.domains  import Project_Setup
from symxplorer.designer_tools.tf_models import Second_Order_BP_TF, cascade_tf

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer")
logger.info("!!! Spicelib_Wrapper imported successfully !!!")

2025-09-23 06:46:35,791 - matplotlib - matplotlib data path: /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data
2025-09-23 06:46:35,805 - matplotlib - CONFIGDIR=/headless/.config/matplotlib
2025-09-23 06:46:35,919 - matplotlib - interactive is False
2025-09-23 06:46:35,919 - matplotlib - platform is linux
2025-09-23 06:46:35,984 - matplotlib - CACHEDIR=/headless/.cache/matplotlib
2025-09-23 06:46:35,990 - matplotlib.font_manager - Using fontManager instance from /headless/.cache/matplotlib/fontlist-v390.json
2025-09-23 06:46:36,519 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-09-23 06:46:36,528 - SymXplorer - !!! Spicelib_Wrapper imported successfully !!!


# Instantiations


In [3]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
logger = setup_loggers()

06:46:36 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
06:46:36 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-09-23_06-46-36.log
06:46:36 - SymXplorer: [INFO] 🔧 spicelib logger set to DEBUG


In [4]:
# s = sp.symbols("s")
# target_tf = (s + 1) / (s**2 + 24*s + 2)
filter_inst = Second_Order_BP_TF(q=10, fc=1e9, k_bp=1e3)
target_tf   = filter_inst.get_tf()
target_tf

200000000000.0*pi*s/(s**2 + 200000000.0*pi*s + 4.0e+18*pi**2)

In [5]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

06:46:36 - SymXplorer.domains: [INFO] Loaded project 'Tunable-TIA' with 8 DUT params and 2 probes.


Project_Setup(project_name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(9.999999999999999e-06), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(9.999999999999999e-06), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(9.999999999999999e-06), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(9.999999999999999e-06

In [6]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.project_name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

06:46:36 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
06:46:36 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:46:36 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
06:46:36 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
06:46:36 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
06:46:36 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
06:46:36 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:46:36 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
06:46:36 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
06:46:36 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
06:46:36 - SymXplorer.spicelib: [INFO] Te

In [7]:
circuit_optimizer = Nevergrad_Spice_Bode_Optimizer(
    spicelib_wrapper=wrapper,
    target_tf=target_tf,
    output_node='vout',
    frequency_weight=Frequency_Weight(lower=1, upper=1e12),
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

# Sanity Check

In [8]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

06:46:36 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
06:46:36 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
06:46:37 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
06:46:37 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
06:46:37 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
06:46:37 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [9]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Log{Cl(0,6,b),exp=2.15},x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 10.000000000000002, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [10]:
circuit_optimizer.optimize()

06:46:37 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 100
Optimizing:   0%|          | 0/100 [00:00<?, ?trial/s]2025-09-23 06:46:37,413 - nevergrad.optimization.optimizerlib - CMA selected CMAbounded optimizer.
06:46:37 - SymXplorer.optimizer: [INFO] computing the target complex response for 200000000000.0*pi*s/(s**2 + 200000000.0*pi*s + 4.0e+18*pi**2)
Optimizing: 100%|██████████| 100/100 [00:33<00:00,  3.02trial/s]


[{'params': {'x_dut_nfet_w': 6.663837325926693,
   'x_dut_nfet_l': 45.02532451074238,
   'x_dut_cap_w': 52.84100144249619,
   'x_dut_cap_l': 54.06514666009644,
   'x_dut_res_s_l': 45.3616689619428,
   'x_dut_res_s_w': 48.53300933368715,
   'x_dut_res_3_l': 47.55381213054275,
   'x_dut_res_3_w': 53.831002388485274},
  'loss': np.float64(28826.63113660768)},
 {'params': {'x_dut_nfet_w': 9.668557630761784,
   'x_dut_nfet_l': 41.11246137495429,
   'x_dut_cap_w': 55.30965703444768,
   'x_dut_cap_l': 53.18305417393835,
   'x_dut_res_s_l': 55.782534354464325,
   'x_dut_res_s_w': 43.37277723282036,
   'x_dut_res_3_l': 50.30480109987716,
   'x_dut_res_3_w': 49.70173488938018},
  'loss': np.float64(76417.68107980624)},
 {'params': {'x_dut_nfet_w': 13.083497103100154,
   'x_dut_nfet_l': 48.311615815735685,
   'x_dut_cap_w': 46.55517250850647,
   'x_dut_cap_l': 49.23010131467345,
   'x_dut_res_s_l': 41.95847326033556,
   'x_dut_res_s_w': 57.62468369459422,
   'x_dut_res_3_l': 46.0185662495325,
   

In [11]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

06:47:11 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
06:47:11 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [12]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss = out

06:47:11 - SymXplorer.optimizer: [INFO] loss: 2143.648270319055
